# Derive gene modules from a single cNMF run on all NB cells at once

In [ ]:
library(ggplot2)
library(RColorBrewer)
library(dplyr)
library(plyr)
library(DescTools)
library(tidyr)


source('../sc_analysis_palettes_and_helpers.r')

wd <- './consensus_modules_R/'

## Load results from cNMF run on NB cells from all datasets at once for chosen k.

In [ ]:
# Load  modules
## Load the Z-scored GEPs which reflect how enriched a gene is in each GEP relative to all of the others

tum_name <- 'allmerged_nrblstm'
k <- 47

gene_scores <- t(read.delim(paste0('./nmf_data/out/',tum_name,'/',tum_name,'.gene_spectra_score.k_',k,'.dt_0_1.txt'), header = T, sep = '\t', row.names = 1))

## Extract top genes contributing to each module.

In [ ]:
# The ranking functions always rate the lowest value first. Invert to start from highest values.
gene_scores <- gene_scores*(-1)
    
# Remove genes that only have NAs
gene_scores <- gene_scores[rowSums(is.na(gene_scores)) != ncol(gene_scores), ]
    
# 1. Get a matrix that rankss the gene contributions to each factor k
geneRanks_forMod <- as.data.frame(colRanks(gene_scores, preserveShape = T))
colnames(geneRanks_forMod) <- colnames(gene_scores)
rownames(geneRanks_forMod) <- rownames(gene_scores)
    
# 2. Get a matrix that ranks for each gene the factors k to which it contributes
modRanks_forGene <- as.data.frame(rowRanks(gene_scores))
colnames(modRanks_forGene) <- colnames(gene_scores)
rownames(modRanks_forGene) <- rownames(gene_scores)
    
mod_list <- list()
    
print(paste0('This is for k = ',k))

# For each module, add genes, starting with the most highly weighted gene.
for(i in 1:ncol(geneRanks_forMod)){

    geneRanks_forMod_i <- geneRanks_forMod[,i]
    names(geneRanks_forMod_i) <- rownames(geneRanks_forMod)
    geneRanks_forMod_i <- sort(geneRanks_forMod_i, decreasing = F)
        
    gene_counter <- 0
        
    for(j in 1:length(geneRanks_forMod_i)){

        # If a gene has a higher rank in another module, flag it!
        if(modRanks_forGene[names(geneRanks_forMod_i)[j],i] != 1){
            gene_counter <- gene_counter + 1
            print(paste0('genecounter ',gene_counter))
            # Allow up to four flagged genes before stopping addition of genes to the final module
            if(gene_counter <= 4){
                    next
            }else{
                mod_list[[i]] <- names(geneRanks_forMod_i)[1:(j-1)]
                print(length(mod_list))
                break
            }
                
        }else{
            next
        }
    }
}   

saveRDS(mod_list, file = paste0(wd, '/', 'mod_list_',tum_name,'_k',k,'_entireDat.rds'))


In [ ]:
# Convert to csv-format

nobs <- sapply(mod_list, length)
seqmax <- seq_len(max(nobs))
mod_mat <- as.data.frame(sapply(mod_list, '[', i = seqmax))
    
write.table(mod_mat, paste0(wd, '/', 'mod_table_',tum_name,'_k',k,'_entireDat.csv'), quote = F, sep = ',')
